In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Kaggle环境准备工作

In [ ]:
!rm -rf /kaggle/working/*

创建accelerate配置文件（T4x2）

In [ ]:
from pathlib import Path
# 创建配置目录
config_dir = Path.home() / ".cache/huggingface/accelerate"
config_dir.mkdir(parents=True, exist_ok=True)

# 使用双卡T4的默认配置创建配置文件内容
config_content = """
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: MULTI_GPU
downcast_bf16: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: no
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: false
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

# 写入配置文件
config_path = config_dir / "default_config.yaml"
with open(config_path, "w") as f:
    f.write(config_content.strip())  # 使用 strip() 移除首尾空行

print(f"配置文件已成功写入: {config_path}")
print(f"内容如下:\n{config_content}")

准备工作

In [ ]:
import PIL

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

from torch.utils.tensorboard import SummaryWriter
import datetime
from zoneinfo import ZoneInfo
# 创建东8区时区对象
tz_east8 = ZoneInfo('Asia/Shanghai')

#======================================================================
# import accelerate
from accelerate import Accelerator
from accelerate.utils import set_seed
# 自定义追踪器
from torch.utils.tensorboard import SummaryWriter
from accelerate.tracking import GeneralTracker, on_main_process
from typing import Union, Optional
#======================================================================


# 0. 自定义追踪器
class MyCustomTracker(GeneralTracker):
    """
    my custom `Tracker` class that supports `tensorboard`. Should be initialized at the start of your script.

    Args:
        run_name (`str`):
            The name of the experiment run
        logging_dir (`str`, `os.PathLike`):
            Location for TensorBoard logs to be stored.
        kwargs:
            Additional key word arguments passed along to the `tensorboard.SummaryWriter.__init__` method.
    """

    name = "custom"
    requires_logging_directory = True

    @on_main_process
    def __init__(self, run_name: str, logging_dir: Union[str, os.PathLike],
                 **kwargs):
        super().__init__()
        self.run_name = run_name
        self.logging_dir = os.path.join(logging_dir, run_name)
        os.makedirs(self.logging_dir, exist_ok=True)
        self.writer = SummaryWriter(self.logging_dir, **kwargs)

    @property
    def tracker(self):
        return self.writer
    
    @on_main_process
    def log(self, values: dict, step: Optional[int], **kwargs):
        # 对每个键值对调用 add_scalar
        if step is None:
            return
        for key, value in values.items():
            self.add_scalar(key, value, global_step=step)
    
    @on_main_process
    def store_init_configuration(self, values: dict):
        """Log experiment configuration parameters"""
        text = "\n".join([f"{k}: {v}" for k, v in values.items()])
        self.add_text(tag="config", text_string=text)

    @on_main_process
    def add_scalar(self, tag, scalar_value, **kwargs):
        self.writer.add_scalar(tag=tag, scalar_value=scalar_value, **kwargs)
    
    @on_main_process
    def add_scalars(self, main_tag, tag_scalar_dict, **kwargs):
        self.writer.add_scalars(main_tag=main_tag, tag_scalar_dict=tag_scalar_dict, **kwargs)

    @on_main_process
    def add_text(self, tag, text_string, **kwargs):
        self.writer.add_text(tag=tag, text_string=text_string, **kwargs)

    @on_main_process
    def add_figure(self, tag, figure, **kwargs):
        self.writer.add_figure(tag=tag, figure=figure, **kwargs)

    @on_main_process
    def finish(self):
        """Close the writer when finished"""
        self.writer.close()

## ResNet网络简介
ResNet即残差网络，它解决的是如何训练较深层网络的问题：
- 较深的网络面临梯度消失和梯度爆炸的问题（通过参数初始化和BatchNorm层来缓解）
- 深度的加深反而会导致训练精度变差

ResNet提出了残差块的概念

![残差块的结构](https://s2.loli.net/2025/07/18/2GcEOWwF9b7mD5K.png)

用 $G(x)$ 表示整个残差块的映射：$G(x)=F(x)+x$。这个结构的好处是：它至少不会比浅层的网络差，因为 $F(x)$ 为0时残差块是恒等映射，深度学习训练的是这个“残差”，由学习来决定是否开启这一层网络。

ResNet借鉴了VGG的思路，先搭建了一个图像大小逐渐减半，深度逐渐翻倍的平坦网络。之后，在平坦网络连续的3x3卷积层上添加残差连接。下图中的实线代表普通的短路连接，虚线表示需要变换张量形状的短路连接。

![ResNet网络结构图](https://s2.loli.net/2025/07/18/Y2zrsjgDCyv6cG5.png)



## 残差块
## 两层残差块（BasicBlock）

残差块有两种形式，分别对应50层以下和50层以上的场景使用。

![两种残差块的形式](https://s2.loli.net/2025/07/18/IhLzkS4EwqviZ6u.png)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channel, out_channel, stride=1, 
                downsample=None, **kwargs):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel,out_channels=out_channel,
            kernel_size=3,stride=stride,padding=1,bias=False    # stride为1时高宽不变，为2时高宽减半。
        )
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(
            in_channels=out_channel,out_channels=out_channel,
            kernel_size=3,stride=1,padding=1,bias=False # 高宽不变
        )
        self.bn2 = nn.BatchNorm2d(out_channel)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

## 三层残差块（Bottlenneck）

In [ ]:

class Bottleneck(nn.Module):
    """
    注意：原论文中，在虚线残差结构的主分支上，第一个1x1卷积层的步距是2，第二个3x3卷积层步距是1。
    但在pytorch官方实现过程中是第一个1x1卷积层的步距是1，第二个3x3卷积层步距是2，
    这么做的好处是能够在top1上提升大概0.5%的准确率。
    可参考Resnet v1.5 https://ngc.nvidia.com/catalog/model-scripts/nvidia:resnet_50_v1_5_for_pytorch
    """
    expansion = 4

    def __init__(self, in_channel, out_channel, stride=1, downsample=None,
                    groups=1, width_per_group=64):
        super(Bottleneck, self).__init__()

        width = int(out_channel * (width_per_group / 64.)) * groups

        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=width,
                                kernel_size=1, stride=1, bias=False)  # squeeze channels
        self.bn1 = nn.BatchNorm2d(width)
        # -----------------------------------------
        self.conv2 = nn.Conv2d(in_channels=width, out_channels=width, groups=groups,
                                kernel_size=3, stride=stride, bias=False, padding=1)
        self.bn2 = nn.BatchNorm2d(width)
        # -----------------------------------------
        self.conv3 = nn.Conv2d(in_channels=width, out_channels=out_channel*self.expansion,
                                kernel_size=1, stride=1, bias=False)  # unsqueeze channels
        self.bn3 = nn.BatchNorm2d(out_channel*self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out

注意到代码里多了`groups`和`width_per_group`参数，在Conv2d中，设置groups可以实现并行计算卷积，提升计算效率。当然，要根据实际情况调整channel，所以有了这两个参数。

## ResNet网络

层数不同的ResNet架构以表格形式展示

![层数不同的ResNet架构参数](https://s2.loli.net/2025/07/18/Ihg7UT2oHqRmCQf.png)

其中Conv3-1、Conv4-1、Conv5-1步幅为二并且启用了 $1 \times 1$ 的卷积调整短路连接的channel数。

In [ ]:

class ResNet(nn.Module):

    def __init__(self, block, blocks_num, num_classes=1000,
                include_top=True, groups=1, width_per_group=64):
        super(ResNet, self).__init__()
        self.include_top = include_top
        self.in_channel = 64

        self.groups = groups
        self.width_per_group = width_per_group

        self.conv1 = nn.Conv2d(3, self.in_channel, kernel_size=7, stride=2,
                                padding=3, bias=False) # 高宽减半
        self.bn1 = nn.BatchNorm2d(self.in_channel)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1) # 高宽减半
        self.layer1 = self._make_layer(block, 64, blocks_num[0])       # Conv2_x的步幅为1（在池化层已经高宽减半）并且不改变channel数
        self.layer2 = self._make_layer(block, 128, blocks_num[1], stride=2) # Conv3_x到Conv5_x的步幅为2（高宽减半）并且channel数翻倍
        self.layer3 = self._make_layer(block, 256, blocks_num[2], stride=2)
        self.layer4 = self._make_layer(block, 512, blocks_num[3], stride=2)
        if self.include_top:
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))  # output size = (1, 1)
            self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        

    def _make_layer(self, block, channel, block_num, stride=1):
        downsample = None
        if stride != 1 or self.in_channel != channel * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channel, channel * block.expansion, kernel_size=1,
                            stride=stride, bias=False),
                nn.BatchNorm2d(channel * block.expansion)
            )

        layers = []
        layers.append(block(self.in_channel,
                            channel,
                            downsample=downsample,
                            stride=stride,
                            groups=self.groups,
                            width_per_group=self.width_per_group))
        self.in_channel = channel * block.expansion

        for _ in range(1, block_num):
            layers.append(block(self.in_channel,
                                channel,
                                groups=self.groups,
                                width_per_group=self.width_per_group))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        if self.include_top:
            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

        return x

def resnet34(num_classes=1000, include_top=True):
    # https://download.pytorch.org/models/resnet34-333f7ec4.pth
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes, include_top=include_top)

在`_make_layer`中，`channel`和`block.expansion`决定最终输出通道数，创建的第一个残差块是`stride`为2的虚线短路连接，所以`stride`为2或者`self.in_channel`不等于`channel * expansion`时加入下采样操作参数，此时输出高宽减半且通道数为`channel * expansion`。第一个块之后，更新了`self.in_channel`为`channel * expansion`，并且`stride`变回默认值1，输出高宽不变，通道数不变。在单个残差块内部，通道数变化会从in_channel->channel->...->channel*expansion。

## 导入数据集

In [ ]:
def create_dataloaders(batch_size=64):
    data_dir = r'/kaggle/input/dogs-vs-cats-for-pytorch/Cat_Dog_data'
    # 数据预处理
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # 加载数据
    train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'train'), transform=train_transform)
    val_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'test'), transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, val_loader

## 导入模型

In [ ]:
def create_net():
    net = resnet34(2)
    return net

## 创建训练循环

In [ ]:
def training_loop(epochs = 10,
                    lr = 1e-3,
                    batch_size= 256,
                    ckpt_path = "checkpoint.pt",
                    mixed_precision="no", #'fp16'
                    ):

    train_dataloader, eval_dataloader = create_dataloaders(batch_size)
    model = create_net()


    optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=10*lr, # 太高会导致梯度爆炸
                                epochs=epochs, steps_per_epoch=len(train_dataloader))

    #======================================================================
    # 初始化 accelerator 并设定种子为 42
    set_seed(42)
    tracker = MyCustomTracker(run_name='ResNet-Training', logging_dir='tensorboard_logs')
    accelerator = Accelerator(
        log_with=tracker,  # 指定使用 TensorBoard
        mixed_precision=mixed_precision
    )

    accelerator.print(f'device {str(accelerator.device)} is used!')
    # 将模型/数据移动到 device
    model, optimizer,lr_scheduler, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer,lr_scheduler, train_dataloader, eval_dataloader)
    #======================================================================

    # 开始循环
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for batch in train_dataloader:
            features,labels = batch
            preds = model(features)
            loss = nn.CrossEntropyLoss()(preds,labels)
            #======================================================================
            #attention here!
            accelerator.backward(loss) #loss.backward()
            #======================================================================

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            # 计算统计量
            train_loss += loss.item()
            _, predicted = preds.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        # 每个epoch记录一次训练指标
        train_accuracy = 100. * correct / total
        current_lr = optimizer.param_groups[0]['lr']
        
        # 验证阶段
        model.eval()
        accurate = 0
        num_elems = 0
        

        for _, batch in enumerate(eval_dataloader):
            features,labels = batch
            with torch.no_grad():
                preds = model(features)
            predictions = preds.argmax(dim=-1)

            #======================================================================
            #gather data from multi-gpus (used when in ddp mode)
            predictions = accelerator.gather(predictions)
            labels = accelerator.gather(labels)
            #======================================================================

            accurate_preds =  (predictions==labels)
            num_elems += accurate_preds.shape[0] # type: ignore
            accurate += accurate_preds.long().sum() # type: ignore

        eval_accuracy = 100. * accurate.item() / num_elems # type: ignore

        # 记录验证指标
        tracker.add_scalars(
            main_tag="Accuracy&Loss", 
            tag_scalar_dict={
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "val_accuracy": eval_accuracy
            }, 
            global_step=epoch
        )

        tracker.add_scalar(
            tag="learning_rate", 
            scalar_value=current_lr, 
            global_step=epoch
        )
            

        #======================================================================
        #print logs and save ckpt
        accelerator.wait_for_everyone()
        nowtime = datetime.datetime.now(tz_east8).strftime('%Y-%m-%d %H:%M:%S')
        accelerator.print(f"epoch【{epoch}】@{nowtime} --> train_accuracy = {train_accuracy:.2f}% | eval_accuracy = {eval_accuracy:.2f}% | train_loss = {train_loss} | lr = {current_lr}")
        if epoch % 15 == 0:
            unwrapped_net = accelerator.unwrap_model(model)
            accelerator.save(unwrapped_net.state_dict(),ckpt_path+"_"+str(epoch))
        #======================================================================

## 开始训练

In [ ]:
from accelerate import notebook_launcher
#args = (5,1e-4,1024,'checkpoint.pt','no')

args = dict(epochs = 5,
        lr = 1e-3,
        batch_size= 128,
        ckpt_path = "checkpoint.pt",
        mixed_precision="fp16").values()
notebook_launcher(training_loop, args, num_processes=2)

将日志打包方便在kaggle上下载

In [ ]:
!zip -r tensorboard_logs.zip tensorboard_logs

训练使用了余弦退火学习率调度策略

![学习率曲线变化](https://s2.loli.net/2025/07/19/OFvr35HBePhIUAg.png)

ResNet准确率能到97.3%

![训练过程](https://s2.loli.net/2025/07/19/fd9BGEnSN2AJwrI.png)